# Irrigation Training — v2.20 TD3 Stage 1 (Kaggle)**Changes from current state:** adds exact n-step returns (Run A) and optionally a critic-leads actor warm-up (Run B). Everything else (actor, critic, reward, eval protocol) is identical to `train_v219b_td3.py`.**Before running:** commit and push the 6 Stage-1 files (`src/rl/`) to your GitHub repo, **or** run the *Write Stage-1 Files* cell below (no push needed).**Disk quota:** 20 GB limit. Replay buffer checkpoints are disabled (`save_replay_buffer=False`), keeping disk use well under 1 GB per run.**Runtime:** ~1–1.5 hr T4 per 250k-step run.

In [ ]:
# ── Set working paths ─────────────────────────────────────────────────────────
import os, sys
from pathlib import Path

REPO        = Path('/kaggle/working/thesis')
RESULTS_DIR = REPO / 'results' / 'rl'
print('REPO:', REPO)

In [ ]:
# ── Clone repo + install deps ─────────────────────────────────────────────────
import subprocess
repo = str(REPO)
if REPO.exists():
    subprocess.run(['rm', '-rf', repo], check=True)
subprocess.run(['git', 'clone', 'https://github.com/taratorbati/thesis.git', repo], check=True)
os.chdir(repo); sys.path.insert(0, repo)
subprocess.run(['pip', 'install', '--quiet',
                'stable-baselines3==2.6.0', 'gymnasium', 'wandb', 'pytest'],
               check=True)
import torch
print(f'PyTorch {torch.__version__}  CUDA={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── WandB (optional) ──────────────────────────────────────────────────────────
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    print('WandB key loaded from Kaggle secrets.')
except Exception:
    print('No WANDB_API_KEY Kaggle secret -- training will log locally only.')

## Write Stage-1 FilesRun this cell if you haven't pushed the files to GitHub yet. If you **have** pushed them, skip to *Verify*.

In [ ]:
# ── Write Stage-1 files ───────────────────────────────────────────────────────
# Run this cell if you have NOT yet pushed the new files to your GitHub repo.
# If you HAVE pushed them, skip this cell (they're already in the cloned repo).
from pathlib import Path
import binascii

_files = {
    'src/rl/nstep_buffer_exact.py': '23207372632f726c2f6e737465705f6275666665725f65786163742e7079202076322e32302e3020202853746167652031290a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a23204558414354206e2d73746570207265706c61792062756666657220666f72206f66662d706f6c6963792054443320286e6f2067616d6d615e312d76732d67616d6d615e6e206861636b292e0a230a232057485920544849532045584953545320287673207468652076322e3130204533207372632f726c2f6e737465705f6275666665722e7079290a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a2320546865204533206275666665722073746f72656420746865206e2d737465702072657475726e20525f6e20627574206c657420534233277320747261696e2829206170706c792067616d6d615e310a2320286e6f742067616d6d615e6e2920746f2074686520626f6f747374726170207465726d2c206265636175736520646f696e672069742022636f72726563746c7922207761732074686f756768740a2320746f2072657175697265206f766572726964696e67205451432e747261696e28292e202049742061646f707465642074686174206173206120646f63756d656e7465640a2320617070726f78696d6174696f6e20287e3720756e697473206f6e2061207e33373820746172676574292e202049742077617320616c736f20626f6c746564206f6e746f20545143202877686f73650a232056444e2d73756d207175616e74696c657320636f6c6c6170736520686572652920616e6420612073746f6368617374696320706f6c69637920287768696368206e6565647320616e0a2320696d706f7274616e63652d73616d706c696e6720636f7272656374696f6e20666f72206d756c74692d73746570206f66662d706f6c6963792072657475726e73202d2d207468650a2320636f6e747269627574696f6e206f662074686520534143286c616d62646129202f205472756e63617465642d5444286c616d626461292070617065722c206e657665720a2320696d706c656d656e746564292e20204e6f6e65206f6620746861742062696e647320666f7220612044455445524d494e4953544943205444332074617267657420706f6c6963792e0a230a23205448452045584143542d67616d6d615e6e20545249434b20286e6f20747261696e2829206f76657272696465206e6565646564290a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a2320546869732062756666657220616363756d756c6174657320746865206e2d737465702072657475726e207769746820697473204f574e20646973636f756e742067616d6d615f626173653a0a232020202020525f6e203d2073756d5f7b6b3d307d5e7b6e2d317d2067616d6d615f626173655e6b202a20725f7b742b6b7d2020202020287472756e636174656420617420616e7920646f6e65290a2320616e642073746f7265732028735f742c20615f742c20525f6e2c20735f7b742b6e206f72207465726d696e616c7d2c20646f6e655f616e79292e0a230a232054686520545241494e4552207365747320746865202a6d6f64656c27732a2067616d6d6120746f2067616d6d615f62617365202a2a206e5f73746570732e202053423327732073746f636b0a232054443320746172676574206973207468656e2c20666f722065766572792073616d706c6564207472616e736974696f6e3a0a232020202020746172676574203d20525f6e202b202831202d20646f6e6529202a206d6f64656c2e67616d6d61202a205128735f7b742b6e7d290a232020202020202020202020203d20525f6e202b202831202d20646f6e6529202a2067616d6d615f626173655e6e202a205128735f7b742b6e7d2920202020202020203c2d2d2045584143540a23205468697320697320636f727265637420626563617573653a0a232020202a206e6f6e2d7465726d696e616c2028646f6e653d30293a2066756c6c2077696e646f772c20646973636f756e742069732065786163746c792067616d6d615f626173655e6e3b0a232020202a207465726d696e616c202020202028646f6e653d31293a20525f6e206973207472756e63617465642061742074686520626f756e6461727920616e642028312d646f6e65293d300a2320202020207a65726f65732074686520626f6f7473747261702c20736f2074686520646973636f756e742076616c756520697320697272656c6576616e742e0a232054686520637269746963207468657265666f7265206c6561726e73207468652067616d6d615f6261736520283d302e39392920646973636f756e7465642072657475726e2c20736f207468650a2320626961735f726174696f20715f707265642d76732d7265616c697365642d72657475726e20646961676e6f73746963207374617973206f6e207468652073616d65207363616c652e0a230a2320574859206e2d737465702041545441434b53205448452076322e323020444956455247454e43450a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a2320546865202d32323220715f7072656420657863757273696f6e20697320626f6f7473747261702d686f72697a6f6e20616d706c696669636174696f6e3a20612039332d737465700a2320657069736f64652061742067616d6d613d302e3939206861732065666665637469766520686f72697a6f6e20312f28312d67616d6d6129207e3d203130302c20616e64207468650a2320312d7374657020746172676574206c65616e73206f6e20746865206372697469632773206f776e2028646976657267696e672920657374696d61746520657665727920737465702e0a23206e2d73746570207265706c61636573206e206f662074686f736520626f6f7473747261702073746570732077697468206e2047524f554e44454420726577617264732c20736872696e6b696e670a23207468652073656c662d7265666572656e7469616c207465726d20746f2067616d6d615e6e20616e642070726f7061676174696e67206120636174617374726f706869632072657761726420746f0a23207468652072656c6576616e74205120696e206f6e652075706461746520696e7374656164206f66206e2e20205468652070726f6a6563742773206f776e2076322e3620286561726c790a23207465726d696e6174696f6e202d3e207e35302d7374657020686f72697a6f6e202d3e207c517c20626f756e64656420666f72203530306b207374657073292076732076322e37202866756c6c0a232039332d7374657020686f72697a6f6e202d3e206361736361646529206973207468652070726f6f66207468617420686f72697a6f6e206c656e67746820697320746865206472697665722e0a230a23205265666572656e6365730a23202d2d2d2d2d2d2d2d2d2d0a232048657373656c20657420616c2e20323031382020225261696e626f77222c2041414149202d2d206e2d737465702072657475726e2061626c6174696f6e2c205365632e332e0a2320466564757320657420616c2e20323032302020202252657669736974696e672046756e64616d656e74616c73206f6620457870657269656e6365205265706c6179222c2049434d4c202d2d0a232020202020202020202020202020202020202020205365632e343a2073746f72652028732c612c525f6e2c735f7b742b6e7d2c646f6e65292c20747261696e20776974682067616d6d615e6e2e0a232046756a696d6f746f20657420616c2e2032303138202241646472657373696e672046756e6374696f6e20417070726f78696d6174696f6e204572726f7220696e204163746f722d4372697469630a232020202020202020202020202020202020202020204d6574686f647322202854443329202d2d2064657465726d696e69737469632074617267657420706f6c6963792e0a232042617274682d4d61726f6e20657420616c2e203230313820284434504729202f20486f7267616e20657420616c2e203230313820284170652d5829202d2d20756e636f727265637465640a23202020202020202020202020202020202020202020736d616c6c2d6e2072657475726e7320617265207374616e646172642026206e6561722d756e62696173656420666f720a2320202020202020202020202020202020202020202064657465726d696e6973746963206f66662d706f6c696379206163746f722d637269746963732e0a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a0a66726f6d205f5f6675747572655f5f20696d706f727420616e6e6f746174696f6e730a0a66726f6d20636f6c6c656374696f6e7320696d706f72742064657175650a66726f6d20747970696e6720696d706f727420416e792c20446963742c204c6973742c20556e696f6e0a0a696d706f7274206e756d7079206173206e700a696d706f727420746f7263682061732074680a66726f6d2067796d6e617369756d20696d706f7274207370616365730a0a66726f6d20737461626c655f626173656c696e6573332e636f6d6d6f6e2e6275666665727320696d706f7274205265706c61794275666665720a0a0a636c617373204e537465705265706c61794275666665724578616374285265706c6179427566666572293a0a202020202222224f66662d706f6c696379206e2d73746570207265706c617920627566666572207769746820616e2065786163742067616d6d615e6e20626f6f74737472617020646973636f756e742e0a0a2020202044726f702d696e20666f7220534233207669612060607265706c61795f6275666665725f636c6173733d4e537465705265706c61794275666665724578616374606020616e640a2020202060607265706c61795f6275666665725f6b77617267733d64696374286e5f73746570733d6e2c2067616d6d613d67616d6d615f626173652960602e202054686520545241494e45520a202020206d7573742073657420746865206d6f64656c277320606067616d6d613d67616d6d615f62617365202a2a206e60602028736565206d6f64756c65206865616465722920736f207468650a2020202073746f636b2054443320746172676574206c696e6520636f6d7075746573206060525f6e202b2028312d646f6e65292a67616d6d615f626173655e6e2a5160602065786163746c792e0a0a20202020506172616d65746572730a202020202d2d2d2d2d2d2d2d2d2d0a202020206e5f7374657073203a20696e740a20202020202020204e756d626572206f6620737465707320746f20636f6d62696e6520286e3e3d31292e20206e3d31207265647563657320746f20746865207374616e64617264206275666665722e0a2020202067616d6d61203a20666c6f61740a2020202020202020446973636f756e74207573656420746f20414343554d554c41544520746865206e2d737465702072657475726e20525f6e2e20204d75737420657175616c207468650a2020202020202020656e7669726f6e6d656e742f72657475726e20646973636f756e74202867616d6d615f626173652c2064656661756c7420302e393929202d2d204e4f54207468650a20202020202020206d6f64656c27732067616d6d61202877686963682074686520747261696e6572207365747320746f2067616d6d615f62617365202a2a206e292e0a202020202222220a0a20202020646566205f5f696e69745f5f280a202020202020202073656c662c0a20202020202020206275666665725f73697a653a20696e742c0a20202020202020206f62736572766174696f6e5f73706163653a207370616365732e53706163652c0a2020202020202020616374696f6e5f73706163653a207370616365732e53706163652c0a20202020202020206465766963653a20556e696f6e5b74682e6465766963652c207374725d203d20226175746f222c0a20202020202020206e5f656e76733a20696e74203d20312c0a20202020202020206f7074696d697a655f6d656d6f72795f75736167653a20626f6f6c203d2046616c73652c0a20202020202020206e5f73746570733a20696e74203d20352c0a202020202020202067616d6d613a20666c6f6174203d20302e39392c0a202020202020202068616e646c655f74696d656f75745f7465726d696e6174696f6e3a20626f6f6c203d20547275652c0a20202020202020202a2a6b77617267733a20416e792c0a20202020293a0a2020202020202020737570657228292e5f5f696e69745f5f280a2020202020202020202020206275666665725f73697a653d6275666665725f73697a652c0a2020202020202020202020206f62736572766174696f6e5f73706163653d6f62736572766174696f6e5f73706163652c0a202020202020202020202020616374696f6e5f73706163653d616374696f6e5f73706163652c0a2020202020202020202020206465766963653d6465766963652c0a2020202020202020202020206e5f656e76733d6e5f656e76732c0a2020202020202020202020206f7074696d697a655f6d656d6f72795f75736167653d6f7074696d697a655f6d656d6f72795f75736167652c0a20202020202020202020202068616e646c655f74696d656f75745f7465726d696e6174696f6e3d68616e646c655f74696d656f75745f7465726d696e6174696f6e2c0a2020202020202020290a2020202020202020696620696e74286e5f737465707329203c20313a0a20202020202020202020202072616973652056616c75654572726f722866226e5f7374657073206d757374206265203e3d20312c20676f74207b6e5f737465707321727d22290a20202020202020206966206e6f742028302e30203c20666c6f61742867616d6d6129203c3d20312e30293a0a20202020202020202020202072616973652056616c75654572726f7228662267616d6d612028525f6e20616363756d756c6174696f6e29206d75737420626520696e2028302c20315d2c20676f74207b67616d6d6121727d22290a202020202020202073656c662e6e5f7374657073203d20696e74286e5f7374657073290a202020202020202073656c662e5f6e5f67616d6d61203d20666c6f61742867616d6d61290a0a202020202020202023204f6e652070656e64696e67204649464f2070657220656e762e2020456e7472793a20286f62732c20616374696f6e2c207265776172642c206e6578745f6f62732c0a20202020202020202320646f6e652c20696e666f292e20205768656e20612077696e646f772069732066756c6c20746865206f6c6465737420656e74727920697320666c757368656420617320616e0a202020202020202023206e2d73746570207472616e736974696f6e3b206f6e20646f6e6520616c6c2072656d61696e696e67202873686f72746572292077696e646f77732061726520666c75736865642e0a202020202020202073656c662e5f70656e64696e673a204c6973745b64657175655d203d205b6465717565282920666f72205f20696e2072616e6765286e5f656e7673295d0a0a2020202023202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a2020202064656620616464280a202020202020202073656c662c0a20202020202020206f62733a206e702e6e6461727261792c0a20202020202020206e6578745f6f62733a206e702e6e6461727261792c0a2020202020202020616374696f6e3a206e702e6e6461727261792c0a20202020202020207265776172643a206e702e6e6461727261792c0a2020202020202020646f6e653a206e702e6e6461727261792c0a2020202020202020696e666f733a204c6973745b446963745b7374722c20416e795d5d2c0a2020202029202d3e204e6f6e653a0a20202020202020202222225175657565206f6e65207472616e736974696f6e2070657220656e763b20666c757368207768656e207468652077696e646f772069732066756c6c206f72206f6e20646f6e652e2222220a2020202020202020666f7220656e765f6920696e2072616e67652873656c662e6e5f656e7673293a0a20202020202020202020202073656c662e5f70656e64696e675b656e765f695d2e617070656e6428280a202020202020202020202020202020206f62735b656e765f695d2e636f707928292c0a20202020202020202020202020202020616374696f6e5b656e765f695d2e636f707928292c0a20202020202020202020202020202020666c6f6174287265776172645b656e765f695d292c0a202020202020202020202020202020206e6578745f6f62735b656e765f695d2e636f707928292c0a20202020202020202020202020202020626f6f6c28646f6e655b656e765f695d292c0a20202020202020202020202020202020696e666f735b656e765f695d2c0a20202020202020202020202029290a2020202020202020202020206966206c656e2873656c662e5f70656e64696e675b656e765f695d29203e3d2073656c662e6e5f73746570733a0a2020202020202020202020202020202073656c662e5f666c7573685f6f6e6528656e765f69290a202020202020202020202020696620626f6f6c28646f6e655b656e765f695d293a0a202020202020202020202020202020207768696c65206c656e2873656c662e5f70656e64696e675b656e765f695d29203e20303a0a202020202020202020202020202020202020202073656c662e5f666c7573685f6f6e6528656e765f69290a0a2020202023202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a20202020646566205f666c7573685f6f6e652873656c662c20656e765f693a20696e7429202d3e204e6f6e653a0a20202020202020202222224275696c6420746865206e2d737465702072657475726e20666f7220746865206f6c646573742070656e64696e67207472616e736974696f6e20616e642073746f72652069742e2222220a202020202020202071203d2073656c662e5f70656e64696e675b656e765f695d0a20202020202020206966206c656e287129203d3d20303a0a20202020202020202020202072657475726e0a0a20202020202020206f6273302c20616374696f6e302c205f2c206e6578745f6f6273302c205f2c20696e666f30203d20715b305d0a0a2020202020202020525f6e203d20302e300a2020202020202020646973636f756e74203d20312e300a202020202020202066696e616c5f6e6578745f6f6273203d206e6578745f6f6273300a202020202020202066696e616c5f646f6e65203d2046616c73650a202020202020202066696e616c5f696e666f203d20696e666f300a0a2020202020202020666f7220285f2c205f2c20726b2c206e6578745f6f62735f6b2c20646f6e655f6b2c20696e666f5f6b2920696e20713a0a202020202020202020202020525f6e202b3d20646973636f756e74202a20726b0a202020202020202020202020646973636f756e74202a3d2073656c662e5f6e5f67616d6d610a20202020202020202020202066696e616c5f6e6578745f6f6273203d206e6578745f6f62735f6b0a20202020202020202020202066696e616c5f646f6e65203d20646f6e655f6b0a20202020202020202020202066696e616c5f696e666f203d20696e666f5f6b0a202020202020202020202020696620646f6e655f6b3a20202020202020202020202020202023207472756e63617465207468652072657475726e2061742074686520657069736f646520626f756e646172790a20202020202020202020202020202020627265616b0a0a2020202020202020232053746f72652065786163746c79206f6e6520286e2d7374657029207472616e736974696f6e207468726f7567682074686520706172656e74206275666665722e0a2020202020202020232043616c6c20737570657228292e61646420286e6f742073656c662e6164642920746f2061766f69642072652d71756575696e67202f20726563757273696f6e2e0a2020202020202020737570657228292e616464280a2020202020202020202020206f6273305b6e702e6e6577617869735d2c0a20202020202020202020202066696e616c5f6e6578745f6f62735b6e702e6e6577617869735d2c0a202020202020202020202020616374696f6e305b6e702e6e6577617869735d2c0a2020202020202020202020206e702e6172726179285b525f6e5d2c2064747970653d6e702e666c6f61743332292c0a2020202020202020202020206e702e6172726179285b666c6f61742866696e616c5f646f6e65295d2c2064747970653d6e702e666c6f61743332292c0a2020202020202020202020205b66696e616c5f696e666f5d2c0a2020202020202020290a2020202020202020712e706f706c65667428290a0a2020202023202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a2020202064656620666c7573685f616c6c5f70656e64696e672873656c6629202d3e204e6f6e653a0a2020202020202020222222466c7573682065766572792070656e64696e67207061727469616c2077696e646f77202863616c6c206265666f726520612062756666657220636865636b706f696e74292e0a0a20202020202020204e6f7420726571756972656420666f7220747261696e696e6720636f72726563746e657373202d2d2070656e64696e6720646174612069732073696d706c790a202020202020202072652d636f6c6c6563746564206166746572206120726573756d65202d2d20627574206b6565707320612073617665642062756666657220636f6d706c6574652e0a20202020202020202222220a2020202020202020666f7220656e765f6920696e2072616e67652873656c662e6e5f656e7673293a0a2020202020202020202020207768696c65206c656e2873656c662e5f70656e64696e675b656e765f695d29203e20303a0a2020202020202020202020202020202073656c662e5f666c7573685f6f6e6528656e765f69290a',
    'src/rl/td3_warmup.py': '23207372632f726c2f7464335f7761726d75702e7079202076322e32302e30202028537461676520312c2052756e2042202264616d70696e6722290a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a23205761726d75704173796d6d65747269634c525444333a20544433207769746820616e206163746f72206c6561726e696e672d72617465205741524d2d55502028226372697469632d6c6561647322292e0a230a2320506c61636520746869732066696c6520696e207372632f726c2f20616c6f6e677369646520747261696e5f76323139625f7464332e70792e2020497420697320612064726f702d696e0a23207265706c6163656d656e7420666f72207468652076322e313962204173796d6d65747269634c525444333a2073616d65206173796d6d65747269632d4c52206f766572726964652c20504c55530a2320616e206f7074696f6e616c2072616d70207468617420686f6c647320746865206163746f72204c52206c6f7720282d3e302920666f72207468652066697273740a2320606163746f725f7761726d75705f7570646174657360206772616469656e74207570646174657320616e64206c696e6561726c792072616973657320697420746f2066756c6c2e2020576974680a23206163746f725f6c725f6d756c74203d3d20312e3020414e44206163746f725f7761726d75705f75706461746573203d3d203020746865206f7665727269646520697320612076657269666965640a23206e6f2d6f7020616e6420746865206d6f64656c20626568617665732065786163746c79206c696b652073746f636b20534233205444332e0a230a2320574859202874686520226c65742074686520637269746963206c65616422206c65766572290a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a232054776f2076657269666965642066616374732061626f7574207468652076322e3230207235202f2052756e2d41202f2030362d3034206469766573206d6f74697661746520746869733a0a23202020312e2054686520715f7072656420646970204f4e53455420747261636b7320606c6561726e696e675f7374617274736020696e20616c6c20666f75722072756e730a23202020202020287e32376b202f207e35316b202f207e32366b202f207e35316b20666f72206c732032352f35302f32352f35306b293a2074686520696e73746162696c6974792069730a2320202020202073656564656420746865206d6f6d656e74206772616469656e74207570646174657320626567696e20616e64207468652064657465726d696e6973746963206163746f720a23202020202020737461727473206d6f76696e67206f6e2061207374696c6c2d756e747261696e6564206372697469632e0a23202020322e2076322e392073686f77656420746865207472616a6563746f7279206973206368616f746963616c6c792073656e73697469766520746f20746865204541524c494553540a232020202020206772616469656e742073746570732028612073696e676c65207065727475726265642073746570206174207e322e326b206368616e6765642074686520626173696e292e0a232041206261726520544433206163746f722077697468206e6f20656e74726f7079207465726d20737072696e747320746f207468652030206d6d20626f756e64617279206265666f7265207468650a2320637269746963206973206120757361626c6520657374696d61746f722028746869732069732065786163746c79207768792076322e3139622064726f7070656420746865203578206163746f720a23204c5220746f203178292e20205761726d696e6720746865206163746f72207570206c657473207468652063726974696320666974206120726561736f6e61626c652076616c75650a23206c616e6473636170652046495253542c20736f20746865206163746f7220636c696d62732061207265616c207375726661636520696e7374656164206f66206e6f697365202d2d207468650a232070726163746963616c20666f726d206f662074686520646561646c792d74726961642072756c652022646f6e277420626f6f74737472617020616e206163746f72206f666620616e0a2320756e63616c69627261746564206372697469632e22202054686520637269746963204c52206973206c6566742061742066756c6c2066726f6d2073746570206f6e652c20736f206f6e6c790a2320746865206163746f722069732064656c617965643b20746869732069732067656e746c657220616e64206d6f7265207461726765746564207468616e206c6f776572696e67207468650a2320676c6f62616c204c5220287768696368207468652070726f6a656374277320224d697374616b6520342220637269746971756520636f72726563746c792063616c6c656420612064656c61790a2320746163746963207468617420706f7374706f6e657320726174686572207468616e2070726576656e7473207468652063617363616465292e0a230a232057485920414e204c522d5343484544554c45204f564552524944452c204e4f5420412043414c4c4241434b0a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a232076322e392773206c6573736f6e3a205342332063616c6c7320605f7570646174655f6c6561726e696e675f72617465602061742074686520746f70206f6620455645525920747261696e28290a232063616c6c20616e642072652d736574732065616368206f7074696d697a65722773204c522066726f6d20746865207363686564756c652c2073696c656e746c7920726576657274696e6720616e790a23204c5220612063616c6c6261636b2077726f7465206265747765656e20757064617465732e2020546865206f7665727269646520686f6f6b20697320746865206f6e6c7920706c61636520616e0a23206163746f722d4c52206368616e67652061637475616c6c7920737469636b732e2020285468652076322e313962204173796d6d65747269634c525444332075736573207468652073616d650a2320686f6f6b20666f72207468652073616d6520726561736f6e2e290a230a23205265666572656e6365730a23202d2d2d2d2d2d2d2d2d2d0a232046756a696d6f746f20657420616c2e203230313820202854443329202d2d2064656c6179656420706f6c69637920757064617465733b20746865206163746f722073686f756c64206c61670a232020202020202020202020202020202020202020202020746865206372697469632e2020546869732067656e6572616c697365732074686174206964656120746f20746865204c522e0a2320476f79616c20657420616c2e203230313720202020202241636375726174652c204c61726765204d696e6962617463682053474422202d2d206c696e656172204c52207761726d2d75702061730a23202020202020202020202020202020202020202020202061207374616e646172642072656d65647920666f72206561726c792d747261696e696e6720696e73746162696c6974792e0a232046756a696d6f746f2f534233204173796d6d65747269634c52544433202874686973207265706f2c20747261696e5f76323139625f7464332e707929202d2d20746865206f766572726964650a2320202020202020202020202020202020202020202020207061747465726e20616e6420746865206d756c743d3d312e30206e6f2d6f702067756172616e7465652e0a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a0a66726f6d205f5f6675747572655f5f20696d706f727420616e6e6f746174696f6e730a0a66726f6d20737461626c655f626173656c696e65733320696d706f7274205444330a0a0a636c617373205761726d75704173796d6d65747269634c5254443328544433293a0a20202020222222544433207769746820616e206173796d6d6574726963206163746f72204c52206d756c7469706c69657220616e6420616e206163746f722d4c52207761726d2d75702072616d702e0a0a20202020436f6e6669677572652076696120636c6173732061747472696275746573204245464f524520636f6e737472756374696f6e20286d6972726f727320686f772076322e31396220736574730a2020202060604173796d6d65747269634c525444332e6163746f725f6c725f6d756c746060293a3a0a0a20202020202020205761726d75704173796d6d65747269634c525444332e6163746f725f6c725f6d756c74202020202020203d20312e30202020202023207374656164792d7374617465206d756c740a20202020202020205761726d75704173796d6d65747269634c525444332e6163746f725f7761726d75705f75706461746573203d2032355f30303020232072616d70206c656e677468202875706461746573290a20202020202020206d6f64656c203d205761726d75704173796d6d65747269634c5254443328706f6c6963793d2e2e2e2c20656e763d2e2e2e2c202e2e2e290a0a2020202053656d616e7469637320286170706c69656420746f20746865204143544f52206f7074696d697a6572206f6e6c792c20657665727920747261696e28292063616c6c2c2041465445520a20202020534233206861732072652d73657420626f7468204c52732066726f6d20746865207363686564756c65202d2d20736f206e6f7468696e6720636f6d706f756e6473293a0a0a20202020202020206163746f725f6c72203c2d207363686564756c655f6c72202a206163746f725f6c725f6d756c74202a206d696e28312c206e5f75706461746573202f207761726d7570290a0a202020202a206163746f725f6c725f6d756c74203d3d20312e3020616e64206163746f725f7761726d75705f75706461746573203d3d203020202d3e206578616374206e6f2d6f702e0a202020202a20637269746963204c52206973206e6576657220746f756368656420286974206c656164732066726f6d2073746570206f6e65292e0a202020202a207468652072616d70206973206b6579656420746f20606073656c662e5f6e5f757064617465736060202863756d756c6174697665206772616469656e742075706461746573292c20736f0a20202020202077697468206772616469656e745f73746570733d312c20747261696e5f667265713d31206974207370616e73207e606163746f725f7761726d75705f757064617465736020656e760a202020202020737465707320696d6d6564696174656c7920616674657220606c6561726e696e675f737461727473602e0a202020202222220a0a20202020232044656661756c747320726570726f647563652073746f636b2054443320286e6f206173796d6d657472792c206e6f207761726d2d7570292e0a202020206163746f725f6c725f6d756c743a20666c6f6174203d20312e300a202020206163746f725f7761726d75705f757064617465733a20696e74203d20300a0a20202020646566205f7570646174655f6c6561726e696e675f726174652873656c662c206f7074696d697a65727329202d3e204e6f6e653a20202320747970653a2069676e6f72655b6f766572726964655d0a202020202020202023203129204c6574205342332073657420424f5448206163746f7220616e6420637269746963204c522066726f6d20746865207363686564756c652e0a2020202020202020737570657228292e5f7570646174655f6c6561726e696e675f72617465286f7074696d697a657273290a0a20202020202020206d756c74203d20666c6f61742873656c662e6163746f725f6c725f6d756c74290a20202020202020207761726d203d20696e742873656c662e6163746f725f7761726d75705f75706461746573290a0a202020202020202023203229204578616374206e6f2d6f70207768656e206e656974686572206665617475726520697320636f6e666967757265642e0a20202020202020206966206d756c74203d3d20312e3020616e64207761726d203c3d20303a0a20202020202020202020202072657475726e0a0a20202020202020202320332920436f6d62696e6564206163746f7220666163746f72203d207374656164792d7374617465206d756c74202a206c696e656172207761726d2d75702072616d702e0a2020202020202020666163746f72203d206d756c740a20202020202020206966207761726d203e20303a0a202020202020202020202020232073656c662e5f6e5f75706461746573207265666c65637473207570646174657320636f6d706c6574656420696e205052494f5220747261696e28292063616c6c730a2020202020202020202020202320287468697320686f6f6b2072756e73206265666f7265207468652063757272656e742063616c6c20696e6372656d656e7473206974292c20736f207468650a202020202020202020202020232072616d70206973206d6f6e6f746f6e653a20302061742074686520666972737420706f73742d6c6561726e696e675f737461727473207570646174652c0a20202020202020202020202023207265616368696e6720606d756c746020616674657220607761726d6020757064617465732e0a202020202020202020202020666163746f72202a3d206d696e28312e302c20666c6f61742873656c662e5f6e5f7570646174657329202f20666c6f6174287761726d29290a0a2020202020202020696620666163746f72203d3d20312e303a0a20202020202020202020202072657475726e0a0a202020202020202023203429204170706c7920746f20746865206163746f72206f7074696d697a6572206f6e6c793b20637269746963204c52207374617973206174207363686564756c652e0a2020202020202020666f7220706720696e2073656c662e6163746f722e6f7074696d697a65722e706172616d5f67726f7570733a0a20202020202020202020202070675b226c72225d203d2070675b226c72225d202a20666163746f720a',
    'src/rl/gym_env_prev_u.py': '23207372632f726c2f67796d5f656e765f707265765f752e7079202076322e32302e302020285374616765203220636f6d706f6e656e74202d2d204f464620647572696e672053746167652031290a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a232049727269676174696f6e456e7650726576553a2049727269676174696f6e456e76202b207468652070726576696f7573206170706c69656420636f6e74726f6c20755f7b742d317d20617320616e0a23206578747261207065722d6167656e74206f62736572766174696f6e20666561747572652e2020506c61636520696e207372632f726c2f20616c6f6e67736964652067796d5f656e762e70792e0a230a232057485920284d61726b6f762d636f6d706c6574656e65737320666f72207468652064656c74612d7520726577617264207235290a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a23207235203d202d616c70686135202a206d65616e5f6e5b202828755f74202d20755f7b742d317d29202f2055425f4d4d295e32205d2070656e616c69736573206461792d746f2d6461790a2320636f6e74726f6c206368616e67652c2062757420755f7b742d317d206973204e4f5420696e207468652062617365206f62736572766174696f6e2e2020412064657465726d696e69737469630a2320626f6f747374726170706564206163746f72207468657265666f72652063616e6e6f742073656520746865207175616e7469747920697473206f776e20736d6f6f74686e6573730a232070656e616c747920646570656e6473206f6e2c20736f20746865204d4450206973206e6f74204d61726b6f7620772e722e742e20723520616e642074686520706f6c6963792063616e6e6f740a23206c6561726e20746f20626520736d6f6f746820696e2074686520776179204d504320697320284d5043206f7074696d69736573207468652077686f6c65207472616a6563746f727920616e640a232068697473206d65616e7c64757c7e3d302e39383b20524c207769746820723520617420616c706861353d302e30303520736174206174207e322e35292e20205374616e64617264206669783a0a232070757420746865206d697373696e67207374617465207661726961626c6520696e746f20746865206f62736572766174696f6e2028537574746f6e202620426172746f2c204d44500a232073746174652073756666696369656e63793b2063662e20616374696f6e2d686973746f7279206175676d656e746174696f6e20696e20636f6e74726f6c20524c292e0a230a23204c41594f55540a23202d2d2d2d2d2d0a232042617365206f627320287573655f6f76657273686f6f745f666561747572653d46616c736529206973206167656e742d6d616a6f723a0a2320202020205b61305f66302e2e61305f66372c2061315f66302e2e61315f66372c202e2e2e2c20613132395f66302e2e613132395f66372c20203c353720676c6f62616c2064696d733e5d0a232054686973207772617070657220696e7365727473206120397468207065722d6167656e7420666561747572652c20707265765f755f6e6f726d203d20636c697028755f7b742d317d2f55425f4d4d2c0a2320302c20312920287a65726f73206f6e2074686520666972737420646179206f66206120736561736f6e2c207768656e205f707265765f6972725f6d6d206973204e6f6e65293a0a2320202020205b61305f66302e2e61305f66372c61305f70726576752c2061315f66302e2e61315f66372c61315f70726576752c202e2e2e2c203c353720676c6f62616c2064696d733e5d0a23204f62732064696d2031303937202d3e20313232372c207065722d6167656e74206665617475726520636f756e742038202d3e20392e0a230a23202a2a2a205245515549524544204d41544348494e47204e4554574f524b204348414e47452028746869732077726170706572206973204e4f542073756666696369656e7420616c6f6e6529202a2a2a0a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a23206e6574776f726b735f7464332e707920686172642d636f64657320746865207065722d6167656e74206665617475726520636f756e7420666f7220424f544820746865206163746f7220616e640a2320746865206372697469633a0a2320202020205444335f4e5f4147454e545f4645415455524553202020203d205632375f4e5f4147454e545f464541545552455320202020203d20382020202020286e6574776f726b735f7464332e7079290a2320202020205444335f5045525f4147454e545f494e5055545f44494d203d205632375f5045525f4147454e545f494e5055545f44494d20203d20363520202020283d2038202b203537290a2320616e64205f5444335368617265644163746f722e5f5f696e69745f5f20617373657274732066656174757265735f64696d203d3d20382a4e202b203537203d20313039372c207468656e0a232072657368617065732066656174757265735b3a2c203a382a4e5d202d3e2028422c204e2c2038292e202046656564696e67207468697320656e76277320313232372d64696d206f627320746f0a23207468652073746f636b206e6574776f726b2077696c6c2028612920747269702074686174206173736572742c206f7220286229206d69732d736c69636520746865206167656e7420626c6f636b2e0a2320546f20414354495641544520707265765f7520796f75206d75737420616c736f2070726f76696465206120392d66656174757265206163746f72202b206372697469632c20652e672e20610a2320736d616c6c2076617269616e7420746861742073657473205f4e5f4147454e545f46454154555245533d3920616e64205f5045525f4147454e545f494e5055545f44494d3d363620283d392b3537290a2320616e6420746865206d61746368696e672066656174757265735f64696d20617373657274202831323237292e2020446f204e4f5420656469742074686520736861726564205632375f2a0a2320636f6e7374616e747320696e206e6574776f726b732e707920696e20706c616365202d2d20746865792061726520726575736564206279207468652076322e31362d76322e3138205341430a232066616d696c792e202054686973206973207468652053746167652d32207461736b3b20756e74696c20697420697320646f6e652c206b656570206578706f73655f707265765f753d46616c73650a2320287468652076322e323020747261696e657220726169736573206120636c656172206572726f722069662069742069732054727565292e0a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a0a66726f6d205f5f6675747572655f5f20696d706f727420616e6e6f746174696f6e730a0a696d706f7274206e756d7079206173206e700a66726f6d2067796d6e617369756d20696d706f7274207370616365730a0a66726f6d207372632e726c2e67796d5f656e7620696d706f72742049727269676174696f6e456e762c204e5f4147454e54532c2055425f4d4d0a0a0a636c6173732049727269676174696f6e456e7650726576552849727269676174696f6e456e76293a0a2020202022222249727269676174696f6e456e76207769746820755f7b742d317d20617070656e6465642061732061207065722d6167656e74206f62736572766174696f6e20666561747572652e2222220a0a20202020646566205f5f696e69745f5f2873656c662c202a617267732c202a2a6b7761726773293a0a2020202020202020737570657228292e5f5f696e69745f5f282a617267732c202a2a6b7761726773290a0a2020202020202020232042617365207065722d6167656e74206665617475726520636f756e7420283820756e6c65737320746865206f76657273686f6f742066656174757265206973206f6e292e0a202020202020202073656c662e5f6e5f666561745f62617365203d20392069662073656c662e5f7573655f6f76657273686f6f745f6665617475726520656c736520380a0a2020202020202020232052656d656d6265722074686520626173652028706172656e742d6275696c7429206f62736572766174696f6e2073706163653b2074686520706172656e7427730a202020202020202023205f6275696c645f6f6273206173736572747320616761696e73742073656c662e6f62736572766174696f6e5f73706163652e73686170652c20736f20776520726573746f72650a2020202020202020232074686973206f6e652061726f756e642074686520737570657228292063616c6c2062656c6f772e0a202020202020202073656c662e5f626173655f6f62735f7370616365203d2073656c662e6f62736572766174696f6e5f73706163650a2020202020202020626173655f64696d203d20696e742873656c662e5f626173655f6f62735f73706163652e73686170655b305d290a0a20202020202020202320457870616e6465642073706163653a202b31206665617475726520706572206167656e742e0a202020202020202073656c662e5f66756c6c5f6f62735f7370616365203d207370616365732e426f78280a2020202020202020202020206c6f773d2d6e702e696e662c20686967683d6e702e696e662c0a20202020202020202020202073686170653d28626173655f64696d202b204e5f4147454e54532c292c2064747970653d6e702e666c6f617433322c0a2020202020202020290a202020202020202073656c662e6f62736572766174696f6e5f7370616365203d2073656c662e5f66756c6c5f6f62735f73706163650a0a20202020646566205f6275696c645f6f62732873656c6629202d3e206e702e6e6461727261793a0a202020202020202023204275696c64207468652062617365206f62736572766174696f6e20756e64657220746865204241534520737061636520736f2074686520706172656e7427730a20202020202020202320696e7465726e616c20736861706520617373657274207061737365732c207468656e20726573746f72652074686520657870616e6465642073706163652e0a202020202020202073656c662e6f62736572766174696f6e5f7370616365203d2073656c662e5f626173655f6f62735f73706163650a20202020202020207472793a0a20202020202020202020202062617365203d20737570657228292e5f6275696c645f6f627328290a202020202020202066696e616c6c793a0a20202020202020202020202073656c662e6f62736572766174696f6e5f7370616365203d2073656c662e5f66756c6c5f6f62735f73706163650a0a20202020202020204e203d204e5f4147454e54530a20202020202020206e66203d2073656c662e5f6e5f666561745f626173650a202020202020202073706c6974203d204e202a206e660a0a20202020202020206167656e745f626c6f636b203d20626173655b3a73706c69745d2e72657368617065284e2c206e66292020202320284e2c206e662920206167656e742d6d616a6f720a202020202020202072657374203d20626173655b73706c69743a5d20202020202020202020202020202020202020202020202020202320353720676c6f62616c2064696d730a0a202020202020202069662073656c662e5f707265765f6972725f6d6d206973204e6f6e653a0a202020202020202020202020707265765f66656174203d206e702e7a65726f7328284e2c2031292c2064747970653d626173652e6474797065290a2020202020202020656c73653a0a202020202020202020202020707265765f66656174203d206e702e636c6970280a202020202020202020202020202020206e702e617361727261792873656c662e5f707265765f6972725f6d6d2c2064747970653d626173652e647479706529202f2055425f4d4d2c20302e302c20312e302c0a202020202020202020202020292e72657368617065284e2c2031290a0a20202020202020206167656e745f626c6f636b203d206e702e636f6e636174656e617465285b6167656e745f626c6f636b2c20707265765f666561745d2c20617869733d312920202320284e2c206e662b31290a20202020202020206f6273203d206e702e636f6e636174656e617465285b6167656e745f626c6f636b2e72657368617065282d31292c20726573745d292e61737479706528626173652e6474797065290a0a2020202020202020617373657274206f62732e7368617065203d3d2073656c662e5f66756c6c5f6f62735f73706163652e73686170652c20280a2020202020202020202020206622707265765f75206f6273207368617065207b6f62732e73686170657d2c206578706563746564207b73656c662e5f66756c6c5f6f62735f73706163652e73686170657d220a2020202020202020290a202020202020202072657475726e206f62730a',
    'src/rl/configs_v220.py': '23207372632f726c2f636f6e666967735f763232302e7079202076322e32302e3020202853746167652031290a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a232053746167652d312072756e20646566696e6974696f6e7320666f7220746865205444332073746162696c69736174696f6e206578706572696d656e74732e2020506c61636520696e0a23207372632f726c2f20616c6f6e677369646520747261696e5f763232305f7464332e70792c20776869636820726561647320434f4e464947535b6e616d655d2e0a230a232044455349474e2052554c453a206368616e6765204f4e45206d616a6f72207468696e67207065722072756e20736f20616e792065666665637420697320617474726962757461626c652e0a2320202052756e2041203d206e2d7374657020414c4f4e452028726f6f742d63617573652066697820666f722074686520626f6f7473747261702d686f72697a6f6e20646976657267656e6365292e0a2320202052756e2042203d2052756e2041202b2074686520544433202264616d70696e6722207061636b61676520286c6f6f702d6761696e20726564756374696f6e292e0a2320426f74682053544152542066726f6d2074686520636f6e66696775726174696f6e2074686174204449564552474544202876322e32302072353a206c6561726e696e675f7374617274730a232035306b2c20723520616374697665292c20736f206120626f756e64656420715f70726564206973206469726563742065766964656e636520746865206368616e67652066697865642069742e0a230a2320574859206e3d35202852756e2041290a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d0a23206e2d73746570207265706c61636573206e20626f6f7473747261702073746570732077697468206e2067726f756e64656420726577617264732c20736872696e6b696e67207468650a232073656c662d7265666572656e7469616c20746172676574207465726d20746f2067616d6d615e6e2e202048657373656c20657420616c2e203230313820285261696e626f772920666f756e640a23206e3d33206f7074696d616c206f6e2041746172693b206e3d35206973206120736c696768746c79206c6f6e67657220686f72697a6f6e2c206a75737469666965642068657265206279207468650a232039332d7374657020736561736f6e20616e64207468652070726f6a6563742773206f776e2065766964656e6365207468617420686f72697a6f6e206c656e677468206973207468650a2320646976657267656e636520647269766572202876322e36207e35302d7374657020686f72697a6f6e3a207c517c20626f756e646564203530306b2073746570733b2076322e372066756c6c0a232039332d7374657020686f72697a6f6e3a2063617363616465292e20206e3d332069732074686520636f6e7365727661746976652066616c6c6261636b206966206e3d35206973206e6f6973792e0a232054686520646973636f756e74206973206170706c6965642045584143544c59202867616d6d615e6e20626f6f747374726170292076696120746865206d6f64656c2d67616d6d6120747269636b0a2320696e20747261696e5f763232305f7464332e7079202b204e537465705265706c61794275666665724578616374202d2d206e6f74207468652067616d6d615e3120617070726f78696d6174696f6e0a23207468652076322e31302045332062756666657220757365642e0a230a2320574859207468652052756e2d422064616d70696e67206b6e6f62732028616c6c205444332773204f574e207374727563747572616c2073746162696c69736572732c206e6f74207468650a2320224d697374616b652034222064656c61792074616374696373206f66206c6f776572696e6720676c6f62616c204c52202f20746175202f20677261642d6e6f726d290a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a23202020706f6c6963795f64656c61792032202d3e20332020203a2046756a696d6f746f20657420616c2e20323031382028544433292064656661756c7420697320323b2064656c6179696e67207468650a232020202020202020202020202020202020202020202020202020206163746f722072656c617469766520746f20746865206372697469632072656475636573206163746f722d7570646174650a2320202020202020202020202020202020202020202020202020202076617269616e63652e20204d696c6420696e637265617365202d3e206d6f72652063726974696320736574746c696e67207065720a232020202020202020202020202020202020202020202020202020206163746f7220737465702e0a232020207461726765745f706f6c6963795f6e6f69736520302e32202d3e20302e332028636c697020302e3520756e6368616e676564293a2054443327732074617267657420736d6f6f7468696e670a232020202020202020202020202020202020202020202020202020202846756a696d6f746f20323031382c207369676d613d302e322f636c69703d302e352920726567756c6172697365732073686172700a232020202020202020202020202020202020202020202020202020205120636f726e6572733b20776964656e696e6720697420666c617474656e732074686520302f31322062616e672d62616e670a2320202020202020202020202020202020202020202020202020202074726f756768732074686174206665656420626f7468207468652070756c73696e6720616e642074686520646976657267656e63652e0a232020202020202020202020202020202020202020202020202020206e6574776f726b735f7464332e7079206e6f7465732074617267657420736d6f6f7468696e672069732054443327730a2320202020202020202020202020202020202020202020202020202064657369676e61746564207265706c6163656d656e7420666f722074686520656e74726f7079207468617420676176652e0a232020206163746f725f7761726d75705f757064617465732030202d3e2032355f3030303a20226372697469632d6c6561647322204c52207761726d2d757020287365650a232020202020202020202020202020202020202020202020202020207464335f7761726d75702e7079292e202054617267657473207468652076657269666965642066696e64696e672074686174207468650a23202020202020202020202020202020202020202020202020202020646970206f6e73657420747261636b73206c6561726e696e675f737461727473202d2d2067697665207468652063726974696320610a2320202020202020202020202020202020202020202020202020202068656164207374617274206265666f7265207468652064657465726d696e6973746963206163746f72206d6f7665732e0a230a23206578706f73655f707265765f752073746179732046616c736520666f7220414c4c206f6620537461676520312028697420697320612053746167652d3220726577617264206368616e676520616e640a23206164646974696f6e616c6c79206e65656473206120392d66656174757265206e6574776f726b3b207365652067796d5f656e765f707265765f752e7079292e202054686520747261696e65720a2320726169736573206966206974206973205472756520776974686f75742074686174206e6574776f726b206368616e67652e0a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a0a66726f6d205f5f6675747572655f5f20696d706f727420616e6e6f746174696f6e730a0a232052657475726e202f20656e7669726f6e6d656e7420646973636f756e74207573656420746f20616363756d756c61746520746865206e2d737465702072657475726e20525f6e20414e442061730a2320746865206261736520666f7220746865206d6f64656c27732067616d6d615e6e20626f6f7473747261702e20204d6174636865732076322e3139622047414d4d412e0a47414d4d415f42415345203d20302e39390a0a0a23202d2d2d2052756e20413a206e2d7374657020616c6f6e65202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a52554e5f41203d2064696374280a202020206c6162656c3d226e7374657035222c0a2020202023206e2d7374657020287468652073696e676c65206368616e67652076732074686520646976657267696e672076322e32302072352072756e290a202020206e5f73746570733d352c0a2020202067616d6d615f626173653d47414d4d415f424153452c0a2020202023206d617463682074686520444956455247494e472072756e20666f7220636c65616e206174747269627574696f6e202876322e3139622064656661756c742069732032355f303030290a202020206c6561726e696e675f7374617274733d35305f3030302c0a202020207265776172645f64755f616c7068613d302e3030352c20202020202020202020202023207235206163746976652c20617320696e2076322e32302072350a2020202023205444332073746162696c69736572732068656c642061742076322e3139622073746f636b20736f206e2d737465702069732069736f6c617465640a20202020706f6c6963795f64656c61793d322c0a202020207461726765745f706f6c6963795f6e6f6973653d302e322c0a202020207461726765745f6e6f6973655f636c69703d302e352c0a202020206163746f725f6c725f6d756c743d312e302c0a202020206163746f725f7761726d75705f757064617465733d302c0a20202020232053746167652d322073776974636820286e65656473206d61746368696e67206e6574776f726b206368616e676529202d2d204f46460a202020206578706f73655f707265765f753d46616c73652c0a290a0a0a23202d2d2d2052756e20423a2052756e2041202b2064616d70696e67207061636b616765202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a2320496e68657269742052756e20412c206f76657272696465206f6e6c79207468652074687265652064616d70696e67206b6e6f62732e0a52554e5f42203d20646963742852554e5f41290a52554e5f422e757064617465280a202020206c6162656c3d226e73746570355f64616d706564222c0a20202020706f6c6963795f64656c61793d332c0a202020207461726765745f706f6c6963795f6e6f6973653d302e332c0a202020206163746f725f7761726d75705f757064617465733d32355f3030302c0a290a0a0a434f4e46494753203d207b0a202020202241223a2052554e5f412c0a202020202242223a2052554e5f422c0a7d0a',
    'src/rl/train_v220_td3.py': '...',
    'src/rl/run_seeds.py': '23207372632f726c2f72756e5f73656564732e7079202076322e32302e30202028537461676520343a207365656420726570726f64756374696f6e203d2074686520616363657074616e63652074657374290a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a232052756e204f4e452066726f7a656e2076322e323020636f6e666967206163726f7373204e20736565647320616e64207265636f7264206120726573756d61626c652063616d706169676e0a23206d616e69666573742e2020506c61636520696e207372632f726c2f20616c6f6e677369646520747261696e5f763232305f7464332e70792e0a230a23205748592028746865207265616c20616363657074616e636520746573742c206e6f74206120766963746f7279206c6170290a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a2320412073696e676c652d7365656420226265617473204d50432220697320616e206172746966616374202d2d207468652070726f6a656374277320224d697374616b65203522207761730a232065786163746c792074686174202876322e37202262656174204d504320696e20342063656c6c73222076616e6973686564206f6e20736565642031292e202054686520736565642073776565700a232069732077686174207475726e7320612063616e64696461746520636f6e66696720696e746f206120646566656e7369626c6520726573756c743a206a7564676520746865204d45414e0a23206163726f737320736565647320616761696e7374207468652053746167652d302073636f7265636172642c20616e6420636f6e6669726d206576657279207365656420636f6e7665726765730a23202866696e616c207e3d20626573742c20715f7072656420626f756e6465642c206775617264206e6576657220747269707329202d2d20776869636820697320616c736f20746865206f6e6c790a23207265616c2074657374206f662077686574686572207468652053746167652d312073746162696c69736174696f6e2061637475616c6c79206669786564207468650a23206d756c746973746162696c697479202873616d6520636f6e6669672c20646966666572656e742064726177206f72646572202d3e2073616d65206f7574636f6d653f292e20204e3d352069730a2320746865206669656c64207374616e646172643b2062756467657420757375616c6c79206166666f72647320332e0a230a232044455349474e0a23202d2d2d2d2d2d0a232020202a20467265657a652074686520636f6e666967206669727374202853746167657320312d33207069636b206974293b2070617373206f6e6c79202d2d636f6e666967202b202d2d73656564732e0a232020202a20526573756d61626c653a206120636f6d706c65746564207365656420697320736b6970706564206f6e20612072652d72756e2c20736f206120637261736865642063616d706169676e0a23202020202072657374617274732077686572652069742073746f70706564202877697468696e2d72756e206372617368207265636f766572792069732074686520747261696e65722773206f776e0a23202020202032356b20636865636b706f696e74733b20746869732064726976657220726573756d65732061742053454544206772616e756c6172697479292e0a232020202a205468652063616d706169676e204a534f4e2069732072657772697474656e20287574662d382920616674657220455645525920736565642c20736f206e6f7468696e67206973206c6f73740a232020202020616e64206e6f7468696e67206e6565647320726572756e6e696e6720746f206b6e6f7720746865207374617475732e0a232020202a204f6e652073656564206661696c696e6720646f6573206e6f742061626f7274207468652063616d706169676e3b2069747320737461747573206973207265636f7264656420616e640a232020202020746865206e65787420736565642070726f63656564732e0a230a232052554e53204e4f5448494e47204f4e20494d504f52542e20204c61756e6368696e67207468697320444f45532073746172742066756c6c20747261696e696e672072756e730a2320287e3120636f6d7075746520756e6974202f203235306b2073746570732065616368292e2020496e766f6b652066726f6d2074686520434c492028736565205f5f6d61696e5f5f292e0a23202d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d2d0a0a66726f6d205f5f6675747572655f5f20696d706f727420616e6e6f746174696f6e730a0a696d706f7274206a736f6e0a66726f6d206461746574696d6520696d706f7274206461746574696d652c2074696d657a6f6e650a66726f6d20706174686c696220696d706f727420506174680a66726f6d20747970696e6720696d706f7274204c6973742c204f7074696f6e616c0a0a66726f6d207372632e726c2e636f6e666967735f7632323020696d706f727420434f4e464947530a0a0a646566205f6e6f772829202d3e207374723a0a2020202072657475726e206461746574696d652e6e6f772874696d657a6f6e652e757463292e69736f666f726d617428290a0a0a6465662072756e5f7365656473280a20202020636f6e6669675f6e616d653a20737472203d202241222c0a2020202073656564733a204f7074696f6e616c5b4c6973745b696e745d5d203d204e6f6e652c0a202020206f75747075745f6469723a20737472203d2022726573756c74732f726c222c0a20202020746f74616c5f74696d6573746570733a204f7074696f6e616c5b696e745d203d204e6f6e652c0a293a0a20202020222222547261696e206060636f6e6669675f6e616d65606020666f722065616368207365656420696e206060736565647360603b207772697465206120726573756d61626c652063616d706169676e2e0a0a2020202052657475726e73207468652063616d706169676e206469637420287b73656564202d3e207b7374617475732c2072756e5f6e616d652c202e2e2e7d7d292e0a202020202222220a202020202320496d706f72746564206865726520286e6f74206174206d6f64756c6520746f702920736f206d6572656c7920696d706f7274696e6720746869732066696c652069732063686561700a202020202320616e6420736964652d6566666563742d667265653b20746865206865617679205342332f746f72636820696d706f72742068617070656e73206f6e6c79206f6e206c61756e63682e0a2020202066726f6d207372632e726c2e747261696e5f763232305f74643320696d706f727420747261696e5f7464335f763232300a0a20202020696620636f6e6669675f6e616d65206e6f7420696e20434f4e464947533a0a20202020202020207261697365204b65794572726f72286622756e6b6e6f776e20636f6e666967207b636f6e6669675f6e616d6521727d3b2063686f696365733a207b736f7274656428434f4e46494753297d22290a202020206966207365656473206973204e6f6e653a0a20202020202020207365656473203d205b302c20312c20325d0a0a202020206c6162656c203d20434f4e464947535b636f6e6669675f6e616d655d5b226c6162656c225d0a202020206f7574203d2050617468286f75747075745f646972290a202020206f75742e6d6b64697228706172656e74733d547275652c2065786973745f6f6b3d54727565290a2020202063616d706169676e5f70617468203d206f7574202f206622736565645f63616d706169676e5f7b636f6e6669675f6e616d657d5f7b6c6162656c7d2e6a736f6e220a0a202020202320526573756d653a206c6f616420616e79207072696f722063616d706169676e20736f20636f6d706c657465642073656564732061726520736b69707065642e0a2020202063616d706169676e3a2064696374203d207b7d0a2020202069662063616d706169676e5f706174682e65786973747328293a0a20202020202020207472793a0a20202020202020202020202063616d706169676e203d206a736f6e2e6c6f6164732863616d706169676e5f706174682e726561645f7465787428656e636f64696e673d227574662d382229290a202020202020202065786365707420457863657074696f6e3a0a20202020202020202020202063616d706169676e203d207b7d0a2020202063616d706169676e2e73657464656661756c742822636f6e6669675f6e616d65222c20636f6e6669675f6e616d65290a2020202063616d706169676e2e73657464656661756c7428226c6162656c222c206c6162656c290a2020202063616d706169676e2e73657464656661756c74282273656564735f726571756573746564222c206c69737428736565647329290a2020202063616d706169676e2e73657464656661756c74282272756e73222c207b7d290a0a20202020646566205f666c75736828293a0a202020202020202063616d706169676e5b22757064617465645f757463225d203d205f6e6f7728290a202020202020202063616d706169676e5f706174682e77726974655f74657874286a736f6e2e64756d70732863616d706169676e2c20696e64656e743d32292c20656e636f64696e673d227574662d3822290a0a202020205f666c75736828290a0a20202020666f7220692c207365656420696e20656e756d6572617465287365656473293a0a20202020202020206b6579203d207374722873656564290a20202020202020207072696f72203d2063616d706169676e5b2272756e73225d2e676574286b65792c207b7d290a20202020202020206966207072696f722e67657428227374617475732229203d3d2022636f6d706c65746564223a0a2020202020202020202020207072696e742866225b72756e5f73656564735d2073656564207b736565647d20616c726561647920636f6d706c65746564202d2d20736b697070696e672e22290a202020202020202020202020636f6e74696e75650a0a202020202020202072756e5f6e616d65203d2066227464335f763232305f7b6c6162656c7d5f736565647b736565647d220a20202020202020207072696e742866225c6e7b2723272a37327d5c6e232073656564207b736565647d2020287b69202b20317d2f7b6c656e287365656473297d292020636f6e666967207b636f6e6669675f6e616d657d20287b6c6162656c7d295c6e7b2723272a37327d22290a202020202020202063616d706169676e5b2272756e73225d5b6b65795d203d207b2272756e5f6e616d65223a2072756e5f6e616d652c2022737461747573223a202272756e6e696e67222c2022737461727465645f757463223a205f6e6f7728297d0a20202020202020205f666c75736828290a0a20202020202020207472793a0a202020202020202020202020747261696e5f7464335f76323230280a20202020202020202020202020202020636f6e6669675f6e616d653d636f6e6669675f6e616d652c0a20202020202020202020202020202020736565643d736565642c0a202020202020202020202020202020206f75747075745f6469723d6f75747075745f6469722c0a20202020202020202020202020202020746f74616c5f74696d6573746570733d746f74616c5f74696d6573746570732c0a202020202020202020202020290a20202020202020202020202063616d706169676e5b2272756e73225d5b6b65795d2e757064617465287374617475733d22636f6d706c65746564222c2066696e69736865645f7574633d5f6e6f772829290a20202020202020206578636570742042617365457863657074696f6e20617320653a202023206b656570207468652063616d706169676e20616c6976653b207265636f726420616e6420636f6e74696e75650a20202020202020202020202063616d706169676e5b2272756e73225d5b6b65795d2e757064617465287374617475733d226661696c6564222c206572726f723d726570722865292c2066696e69736865645f7574633d5f6e6f772829290a2020202020202020202020207072696e742866225b72756e5f73656564735d2073656564207b736565647d204641494c45443a207b6521727d202d2d20636f6e74696e75696e6720746f206e65787420736565642e22290a202020202020202066696e616c6c793a0a2020202020202020202020205f666c75736828290a0a20202020646f6e65203d2073756d283120666f72207620696e2063616d706169676e5b2272756e73225d2e76616c756573282920696620762e67657428227374617475732229203d3d2022636f6d706c6574656422290a202020207072696e742866225c6e5b72756e5f73656564735d2063616d706169676e20636f6d706c6574653a207b646f6e657d2f7b6c656e287365656473297d2073656564732066696e69736865642e20220a2020202020202020202066224d616e69666573743a207b63616d706169676e5f706174687d22290a2020202072657475726e2063616d706169676e0a0a0a6966205f5f6e616d655f5f203d3d20225f5f6d61696e5f5f223a0a20202020696d706f72742061726770617273650a20202020706172736572203d2061726770617273652e417267756d656e74506172736572280a20202020202020206465736372697074696f6e3d280a2020202020202020202020202253746167652d34207365656420726570726f64756374696f6e3a20747261696e206f6e652066726f7a656e2076322e323020636f6e666967206163726f7373204e20220a20202020202020202020202022736565647320616e64207772697465206120726573756d61626c652063616d706169676e206d616e69666573742e204a7564676520746865204d45414e2076732074686520220a2020202020202020202020202253746167652d302073636f7265636172643b20636f6e6669726d206576657279207365656420636f6e7665726765732e220a2020202020202020290a20202020290a202020207061727365722e6164645f617267756d656e7428222d2d636f6e666967222c20202020202020202020747970653d7374722c2064656661756c743d2241222c2063686f696365733d736f7274656428434f4e4649475329290a202020207061727365722e6164645f617267756d656e7428222d2d7365656473222c2020202020202020202020747970653d696e742c206e617267733d222b222c2064656661756c743d5b302c20312c20325d290a202020207061727365722e6164645f617267756d656e7428222d2d6f75747075742d646972222c202020202020747970653d7374722c2064656661756c743d22726573756c74732f726c22290a202020207061727365722e6164645f617267756d656e7428222d2d746f74616c2d74696d657374657073222c20747970653d696e742c2064656661756c743d4e6f6e65290a2020202061726773203d207061727365722e70617273655f6172677328290a0a2020202072756e5f7365656473280a2020202020202020636f6e6669675f6e616d653d617267732e636f6e6669672c0a202020202020202073656564733d617267732e73656564732c0a20202020202020206f75747075745f6469723d617267732e6f75747075745f6469722c0a2020202020202020746f74616c5f74696d6573746570733d617267732e746f74616c5f74696d6573746570732c0a20202020290a',
}
for relpath, hex_content in _files.items():
    p = Path(REPO) / relpath
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(bytes.fromhex(hex_content))
    print(f'  written: {p.relative_to(REPO)} ({p.stat().st_size:,} bytes)')
print('Stage-1 files written.')

## Verify Stage-1 Files

In [ ]:
# ── Verify Stage-1 files are present ─────────────────────────────────────────
from pathlib import Path
needed = [
    'src/rl/nstep_buffer_exact.py',
    'src/rl/td3_warmup.py',
    'src/rl/gym_env_prev_u.py',
    'src/rl/configs_v220.py',
    'src/rl/train_v220_td3.py',
    'src/rl/run_seeds.py',
]
ok = True
for f in needed:
    p = Path(REPO) / f
    mark = 'OK' if p.exists() else 'MISSING - run the Write-Files cell'
    print(f'  {mark}  {f}')
    if not p.exists(): ok = False
if ok:
    print('All Stage-1 files present.')

## Smoke Test

In [ ]:
# ── Smoke test: validate imports + 1k-step pilot ─────────────────────────────
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pytest', 'tests/', '-x', '-q',
                        '--tb=short'], cwd=REPO, capture_output=True, text=True)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError('Smoke tests FAILED — fix before training.')
print('Smoke tests passed.')

## Run A — n-step alone (start here)

In [ ]:
# ── Run A: exact n-step alone ────────────────────────────────────────────────
# Change from current state: adds NStepReplayBufferExact (n=5, exact gamma^n)
# with model.gamma = 0.99^5. Everything else is v2.19b stock. ~1 hr T4.
SEED = 0   # change per session; 0/1/2 for Stage 4 sweep
import subprocess, sys
r = subprocess.run(
    [sys.executable, '-m', 'src.rl.train_v220_td3',
     '--config', 'A', '--seed', str(SEED),
     '--output-dir', str(RESULTS_DIR),
     '--wandb-project', 'sac-irrigation-thesis',
    ],
    cwd=REPO
)
if r.returncode != 0:
    print('Training exited with non-zero code:', r.returncode)
else:
    print('Run A complete.')

## Run B — n-step + damping (only if Run A q_pred dives)

In [ ]:
# ── Run B: n-step + damping (run ONLY if Run A still shows a deep q_pred dip)
# Additional changes: policy_delay 2->3, target_policy_noise 0.2->0.3,
# actor-LR warm-up over first 25k gradient updates (critic-leads).
SEED = 0
import subprocess, sys
r = subprocess.run(
    [sys.executable, '-m', 'src.rl.train_v220_td3',
     '--config', 'B', '--seed', str(SEED),
     '--output-dir', str(RESULTS_DIR),
     '--wandb-project', 'sac-irrigation-thesis',
    ],
    cwd=REPO
)
if r.returncode != 0:
    print('Training exited:', r.returncode)
else:
    print('Run B complete.')

## Stage-1 Gate: Read Telemetry

In [ ]:
# ── Stage-1 gate: read bias_ratio (q_pred) and collapse-guard telemetry ───────
# SUCCESS: q_pred bounded (never below -30ish), eval reward improving,
#          final reward ~= best, guard never trips.
# If q_pred still dives to -100+: run Run B.
import glob, pandas as pd, os
RUN_LABEL = 'nstep5'   # or 'nstep5_damped' for Run B
pattern = str(RESULTS_DIR / f'td3_v220_{RUN_LABEL}_seed{SEED}' / 'bias_ratio_log.csv')
files = glob.glob(pattern)
if not files:
    print('No bias_ratio_log.csv found yet.')
else:
    df = pd.read_csv(files[0])
    print('q_pred summary:')
    print(df[['step','q_pred_mean']].to_string(index=False))
    print()
    # Collapse guard
    guard_f = pattern.replace('bias_ratio_log', 'collapse_guard_log')
    if os.path.exists(guard_f):
        gdf = pd.read_csv(guard_f)
        trips = gdf[gdf.get('aborted', gdf.get('action', '')) == True] if 'aborted' in gdf.columns else gdf[gdf.iloc[:,-1]==True]
        print(f'Collapse guard trips: {len(trips)}  (target: 0)')

## Archive Results

In [ ]:
# ── Archive results ───────────────────────────────────────────────────────────
# /kaggle/working/ output is downloadable from the Output tab.
# To persist across sessions, save to a Kaggle dataset (Dataset > New > from output).
import shutil, datetime
SEED = 0; RUN_LABEL = 'nstep5'   # match what you ran above
src = RESULTS_DIR / f'td3_v220_{RUN_LABEL}_seed{SEED}'
dst = REPO / 'archived_runs' / (src.name + '_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
shutil.copytree(str(src), str(dst),
                ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
print('Archived to:', dst)
print('Download from the Kaggle Output tab, or save as a Kaggle dataset for persistence.')